# Vorstudie: MSE vs. SPO+ mit linearem Modell (PyEPO, Shortest Path)

**Aufbau** (Code in [prestudy.py](../src/prestudy.py))
- 5×5-Gitter (40 Kanten), p = 5 Features, PyEPO-Standardgenerator (zufälliges $B$)
- Raster: deg ∈ {1, 2, 4, 6} × noise ∈ {0, 0.5} × n ∈ {100, 1000}
- m = 5 Refits auf disjunkten Trainingsblöcken aus *einem* `genData`-Aufruf je (deg, noise), damit $B$ und die Testmenge ($n_{test}$ = 1000) gleich bleiben
- Modell: `nn.Linear(5, 40)` ohne Regularisierung; Adam, Lernrate 0,01 linear fallend auf 0, 3000 Schritte, Batch 32 – identisch für MSE und SPO+ (Gurobi als Solver)
- PFI: K = 10 Permutationen je Feature; beide Losses eines Refits sehen dieselben Permutationen (gepaarter Vergleich)
- Maße: ΔMSE roh, ΔMSE skalenbereinigt ($a + s\,\hat c$ auf den Trainingsdaten gefittet, entscheidungsneutral), Δ normierter Regret
- Referenz: Orakel $E[c \mid x]$

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import prestudy as v

CSV = Path("../results/prestudy_no_planted_effects.csv")
if not CSV.exists():
    v.run().to_csv(CSV, index=False)  # ca. 10 min
df = pd.read_csv(CSV)
pd.set_option("display.precision", 4)

Welche Features wirken auf wie viele der 40 Kanten? ($B$ ist der erste Zug aus `RandomState(0)` in `genData`.)

In [2]:
B = np.random.RandomState(0).binomial(1, 0.5, (40, 5))
pd.Series(B.sum(0), index=[f"x{j + 1}" for j in range(5)], name="Kanten")

x1    22
x2    19
x3    22
x4    27
x5    16
Name: Kanten, dtype: int64

## 1 Modellgüte auf der Testmenge

Mittelwerte über die 5 Refits. `mse_adj` ist der MSE nach der entscheidungsneutralen Anpassung $a + s\,\hat c$.

In [ ]:
(v.performance(df).query("loss != 'oracle'")
 .groupby(["noise", "n", "deg", "loss"])[["mse", "mse_adj", "regret"]].mean()
 .unstack("loss"))

In [ ]:
v.plot_over_deg(df, "regret", "normierter Regret")
v.plot_over_deg(df, "mse_adj", "MSE (skalenbereinigt)")

## 2 PFI je Feature

Balken: Mittelwert über die Refits mit 95-%-KI; graue Striche: Orakel. Standard ist n = 1000, noise = 0.5; die Parameter lassen sich ändern.

In [ ]:
v.plot_pfi(df)

## 3 H1: Rangfolge nach ΔMSE vs. ΔRegret

Kendall's τ zwischen den Rangfolgen der 5 Features nach ΔMSE (skalenbereinigt) und nach ΔRegret, je Modell und Refit berechnet, dann über die Refits gemittelt. Bei 5 Features nimmt τ nur 11 Werte an.

In [ ]:
v.kendall_tau(df)

## 4 Loss-Effekt je Feature: gepaarte Differenz SPO+ − MSE

Je Refit $D_j = \mathrm{PFI}_j(\text{SPO+}) - \mathrm{PFI}_j(\text{MSE})$, Mittelwert und 95-%-t-Intervall über die 5 Refits. `*` = Intervall schließt 0 aus (ohne Holm-Korrektur, nur als Pilot).

In [ ]:
v.paired_diff(df).unstack("deg").loc[(0.5, 1000)]

Alle Einstellungen als Tabelle:

In [ ]:
v.paired_diff(df)